特定健診を非侵襲・侵襲にわけ、これを使ってage model作成。metageモデルと比較することで、metabolomeを元にするaging modelの性能評価や解釈性の強みを検証する

In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import japanize_matplotlib
%matplotlib inline
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
from IPython.display import display
import time

import sys
from decimal import Decimal, ROUND_HALF_UP
import re
#For exporting .pdf file with editable text
import matplotlib
matplotlib.rcParams['pdf.fonttype']=42
matplotlib.rcParams['ps.fonttype']=42

import collections

#!pip install MissForest
from missforest import MissForest
from sklearn.ensemble import RandomForestRegressor
import gc
from sklearn.impute import SimpleImputer

from sklearn.preprocessing import StandardScaler, PowerTransformer
from sklearn.decomposition import PCA

#!conda install -c conda-forge -y watermark
%load_ext watermark
%watermark -a 'Akari FUJIMOTO' -udnt -vmw -iv

In [ ]:
from scipy.stats import pearsonr

In [ ]:
japanize_matplotlib.japanize()

print(plt.rcParams['font.family'])

In [ ]:
import math

# 1. import specific test

In [ ]:
#Import dataframe
fileDir = './ExportData/Preprocess/specific/'
ipynbName = '260924_TMM-BAge_PreprocessMetDF-CommCohort67K-ver1_'
fileName = 'specificDF.tsv'
tempDF = pd.read_csv(fileDir+ipynbName+fileName, sep='\t')
tempDF = tempDF.set_index('ParticipantID')

display(tempDF)
display(tempDF.describe(include='all'))
specificDF=tempDF

In [ ]:
spe_dict = {
    'anthopometry': [
        #anthopometry
        '身長 [cm]',
        '体重 [kg]',
        '腹囲 [cm]',
        'BMI [kg/m^2]'],

    'blood_pressure':[
        #cardiovascular
        '収縮期血圧 1回目 [mmHg]',
        '収縮期血圧 2回目 [mmHg]',
        '拡張期血圧 1回目 [mmHg]',
        '拡張期血圧 2回目 [mmHg]'],

    
    'hematopology':[
        '赤血球数 [10^4/μL]',
        '血色素量 [g/dL]',
        'ヘマトクリット値 [%]',
        '白血球数 [/μL]',
        '血小板数 [10^4/μL]',
        'リンパ球 [%]',
        '単球 [%]',
        '好酸球 [%]',
        '好塩基球 [%]',
        '好中球 [%]'],
    
    'clinical_chemistry': [
        #metabolism
        'HbA1c（NGSP） [%]',
        'グルコース [mg/dL]',
        '総コレステロール',
        '中性脂肪 [mg/dL]',
        'HDLコレステロール [mg/dL]',
        'LDLコレステロール（直接法） [mg/dL]',
        #liver
        'AST（GOT） [U/L]',
        'ALT（GPT） [U/L]',
        'γ-GTP [U/L]'],

    'kidney':[
        #renal(maybe from blood)
        '尿素窒素 [mg/dL]',
        '尿酸 [mg/dL]',
        'クレアチニン（血中） [mg/dL]',
        'GFR区分（クレアチニン）',
        #renal
        '尿蛋白',
        '尿糖',
        '尿潜血'],

    'inspection_conditions':[
        '食後区分']}


In [ ]:
#Import dataframe
fileDir = './ExportData/Preprocess/metabolome/'
ipynbName = '260924_TMM-BAge_PreprocessMetDF-CommCohort67K-ver1_'
fileName = 'demographicsDF_2k.tsv'
tempDF = pd.read_csv(fileDir+ipynbName+fileName, sep='\t')

display(tempDF)
display(tempDF.describe(include='all'))
cohortDF=tempDF

In [ ]:
tempDF = cohortDF.copy()
tempDF = tempDF.set_index('ParticipantID')

display(tempDF)

cohortDF = tempDF

# 2. check specific test

In [ ]:
tempDF = cohortDF
variable = 'Age'
xaxis_label = 'Age [years]'
tempD = {'All':'k', 'Female':'tab:red', 'Male':'tab:blue'}

#Check distribution
display(tempDF.groupby('Sex')[variable].describe())
sns.set(style='ticks', context='talk')
plt.figure(figsize=(4, 3))
##Group
for group in tempD.keys():
    if group=='All':
        tempS = tempDF[variable].dropna()
    else:
        tempS = tempDF[variable].loc[tempDF['Sex']==group].dropna()
    print(group+': skewness =', stats.skew(tempS))
    var_label = group+' ('+r'$n$'+' = '+f'{len(tempS):,}'+')'
    sns.distplot(tempS, label=var_label, color=tempD[group])
    plt.axvline(x=tempS.mean(), **{'linestyle':'--', 'color':tempD[group]})
sns.despine()
plt.ylabel('Density')
plt.xlabel(xaxis_label)
plt.legend(bbox_to_anchor=(1, 0.5), loc='center left', borderaxespad=1, labelspacing=0.25)
plt.show()

In [ ]:
tempDF = cohortDF
variable = 'BMI'
xaxis_label = 'BMI [kg m'+r'$^{-2}$'+']'
tempD = {'All':'k', 'Female':'tab:red', 'Male':'tab:blue'}

#Check distribution
display(tempDF.groupby('Sex')[variable].describe())
sns.set(style='ticks',  context='talk')
plt.figure(figsize=(4, 3))
##Group
for group in tempD.keys():
    if group=='All':
        tempS = tempDF[variable].dropna()
    else:
        tempS = tempDF[variable].loc[tempDF['Sex']==group].dropna()
    print(group+': skewness =', stats.skew(tempS))
    var_label = group+' ('+r'$n$'+' = '+f'{len(tempS):,}'+')'
    sns.distplot(tempS, label=var_label, color=tempD[group])
    plt.axvline(x=tempS.mean(), **{'linestyle':'--', 'color':tempD[group]})
sns.despine()
plt.ylabel('Density')
plt.xlabel(xaxis_label)
plt.legend(bbox_to_anchor=(1, 0.5), loc='center left', borderaxespad=1, labelspacing=0.25)
plt.show()

# 3. filter

In [ ]:
tempD2={}
tempDF = specificDF.copy()

for category, features in spe_dict.items():

    features = [col for col in features if col in tempDF.columns]

    resultDF = pd.DataFrame({'Feature': features,
                             'N': len(tempDF),
                             'Non-missing N': [tempDF[col].notna().sum() for col in features],
                            'missing N': [tempDF[col].isna().sum() for col in features]})
    resultDF=resultDF.sort_values('Non-missing N',ascending=False)

    print(f'===== {category} =====')
    display(resultDF)

In [ ]:
na_rate = specificDF.isna().mean(axis=1)

print(na_rate.describe())

plt.hist(na_rate, bins=50)
plt.xlabel('Missing rate')
plt.ylabel('N')
plt.show()

specificDFの中で、欠損率が特定の範囲内に入る参加者が何人いるかの図

# 4. transform to less skew

# -1. check

In [ ]:
tempDF = specificDF.copy()

#Check skewness
tempL = []
for col_n in tempDF.columns:
    tempS = tempDF[col_n].dropna()
    tempL.append(stats.skew(tempS))
tempS = pd.Series(tempL, index=tempDF.columns, name='Skewness')
tempS = tempS.sort_values(ascending=True)

display(tempS)
display(tempS.describe())

sns.set(style='ticks', context='talk')
plt.figure(figsize=(4, 3))
sns.distplot(tempS)
sns.despine()
plt.ylabel('Density')
plt.xlabel('Skewness')
plt.show()

In [ ]:
#Check negative or zero value prior to transformation
print('Minimum:', tempDF.min(axis=None))

In [ ]:
tempDF = specificDF.copy()

for col in tempDF.columns:
    print(col)
    print('n_unique:', tempDF[col].nunique())
    print('values:', tempDF[col].dropna().unique()[:20])

In [ ]:
tempDF.nunique().sort_values()

食後区分、尿蛋白と尿糖は離散値なのでは  
＋と＋＋と＋＋＋の間隔は一定ではないので、便宜上の順序尺度として用いるのは不適切。  
dummy変数化して処理する

In [ ]:
#check vs pregnancy metabolites
sns.histplot(specificDF['食後区分'], kde=True)

In [ ]:
#check vs pregnancy metabolites
sns.histplot(specificDF['尿糖'], kde=True)

In [ ]:
display(specificDF['尿糖'])
display(specificDF['尿糖'].describe(include='all'))


	回答コード	回答文章  
110	1	( - )  
111	2	( + - )  
112	3	( + )  
113	4	( ++ )  
114	5	( +++ )  

In [ ]:
display(specificDF['尿蛋白'])
display(specificDF['尿蛋白'].describe(include='all'))


	回答コード	回答文章  
100	1	( - )  
101	2	( + - )  
102	3	( + )  
103	4	( ++ )  
104	5	( +++ )  

## -2. transform skew about continuous

In [ ]:
tempDF = specificDF.copy()

#continuous only
continuous_cols = [c for c in tempDF.columns
    if not c.startswith(('尿蛋白', '尿糖','食後区分'))]


tempL = []
for col_n in continuous_cols:
    tempS = tempDF[col_n].dropna()
    tempL.append(stats.skew(tempS))
tempS = pd.Series(tempL, index=continuous_cols, name='Skewness')
tempS = tempS.sort_values(ascending=True)

display(tempS)
display(tempS.describe())

sns.set(style='ticks',  context='talk')
plt.figure(figsize=(4, 3))
sns.distplot(tempS)
sns.despine()
plt.ylabel('Density')
plt.xlabel('Skewness')
plt.show()

In [ ]:
tempDF = specificDF.copy()
#continuous only
continuous_cols = [c for c in tempDF.columns
    if not c.startswith(('尿蛋白', '尿糖','食後区分'))]


pt = PowerTransformer(method='yeo-johnson', standardize=False)
skew_cutoff = 1.5

#Log-transformation if skewness is over cutoff
log_cols = []
for col_n in continuous_cols:
    tempS = tempDF[col_n].dropna()
    skewness = stats.skew(tempS)
    
    if skewness > skew_cutoff:
        tempD = tempDF[col_n].notna()
        tempDF.loc[tempD, col_n] =  pt.fit_transform(tempDF.loc[tempD, col_n].to_numpy()
            .reshape(-1, 1)).ravel()
        log_cols.append(col_n)

print('Yeo-Johnson transformed columns:')
print(log_cols)

display(tempDF)
display(tempDF.describe(include='all'))

skewDF = tempDF#Update

In [ ]:
tempDF = skewDF

#Check skewness
tempL = []
for col_n in continuous_cols:
    tempS = tempDF[col_n].dropna()
    tempL.append(stats.skew(tempS))
tempS = pd.Series(tempL, index=continuous_cols, name='Skewness')
tempS = tempS.sort_values(ascending=True)

display(tempS)
display(tempS.describe())

sns.set(style='ticks',  context='talk')
plt.figure(figsize=(4, 3))
sns.distplot(tempS)
sns.despine()
plt.ylabel('Density')
plt.xlabel('Skewness')
plt.show()

# 6.random forest imputation

In [ ]:
tempDF = skewDF

#Check
count = tempDF.isnull().to_numpy().sum(axis=None)
nrows, ncols = tempDF.shape
print('nrows:', nrows)
print('ncols:', ncols)
print('nrows*ncols:', nrows*ncols)
print('nNaNs:', count, '(', count/(nrows*ncols)*100, '[%])')
print('Mean nNaNs per row:', count/nrows)
print('Mean nNaNs per column:', count/ncols)

In [ ]:
#腑分け
tempDF = skewDF.copy()

cat_cols = ['尿蛋白', '尿糖', '食後区分']

num_cols = [
    c for c in tempDF.columns
    if c not in cat_cols]

In [ ]:
for name, cols in [('Continuous', continuous_cols),
                   ('Categorical', cat_cols)]:

    n_nan = tempDF[cols].isna().sum().sum()
    n_cell = len(tempDF) * len(cols)

    print(f'\n[{name}]')
    print('ncols:', len(cols))
    print('nNaNs:', n_nan)
    print('NaN rate:', n_nan / n_cell * 100, '[%]')
    print('Mean nNaNs per row:', n_nan / len(tempDF))
    print('Mean nNaNs per column:', n_nan / len(cols))


display(
    tempDF[cat_cols].isna().sum()
        .to_frame('nNaN')
        .assign(NaN_rate=lambda x: x['nNaN'] / len(tempDF) * 100))

In [ ]:
# Imputation: skewDF -> skewDF_RF
# Continuous: MissForest
# Categorical: Most frequent imputation

tempDF = skewDF.copy()

t_start = time.time()

print('===============================================================')
print('Imputation start:', time.ctime(t_start))
print('Input shape:', tempDF.shape)
print('Initial NaN:', tempDF.isna().sum().sum())

# 1. Columns
num_cols_temp = [c for c in num_cols if c in tempDF.columns]
cat_cols_temp = [c for c in cat_cols if c in tempDF.columns]

print('Continuous variables:', len(num_cols_temp))
print('Categorical variables:', len(cat_cols_temp))

# 分類されていない列があれば確認
unclassified_cols = [
    c for c in tempDF.columns
    if c not in num_cols_temp and c not in cat_cols_temp]

if unclassified_cols:
    print('WARNING: Unclassified columns:', unclassified_cols)

skewDF_RF = tempDF.copy()

# 2. Continuous variables: MissForest
if num_cols_temp:
    xDF_num = tempDF[num_cols_temp].copy()

    print('---------------------------------------------------------------')
    print('Continuous imputation start:', time.ctime())
    print('NaN before RF:', xDF_num.isna().sum().sum())

    if xDF_num.isna().any().any():
        # MissForest用に列名を単純化
        original_num_cols = xDF_num.columns.tolist()
        safe_cols = [f'Num_{i}' for i in range(len(original_num_cols))]
        xDF_num.columns = safe_cols

        imputer = MissForest(
            categorical=None,
            max_iter=5,
            rgr=RandomForestRegressor(
                n_estimators=150,
                criterion='squared_error',
                max_features='sqrt',
                max_depth=None,
                n_jobs=-1,
                random_state=123))

        t_rf = time.time()
        tempA = imputer.fit_transform(xDF_num)

        # 元の列名に戻す
        if isinstance(tempA, pd.DataFrame):
            tempA = tempA.to_numpy()
    
        t_rf_elapsed = time.time() - t_rf
        xDF_num = pd.DataFrame(
            data=tempA,
            index=xDF_num.index,
            columns=original_num_cols)

        print(
            'RF elapsed time:',
            round(t_rf_elapsed // 60), 'min',
            round(t_rf_elapsed % 60, 1), 'sec')

    else:
        print('No missing values in continuous variables; skipped RF.')

    skewDF_RF.loc[:, num_cols_temp] = xDF_num

# 3. Categorical variables: Most frequent imputation
if cat_cols_temp:
    xDF_cat = tempDF[cat_cols_temp].copy()
    print('---------------------------------------------------------------')
    print('Categorical imputation start:', time.ctime())
    print('NaN before categorical imputation:', xDF_cat.isna().sum().sum())

    if xDF_cat.isna().any().any():
        cat_imputer = SimpleImputer(strategy='most_frequent')

        xDF_cat = pd.DataFrame(
            cat_imputer.fit_transform(xDF_cat),
            index=xDF_cat.index,
            columns=cat_cols_temp)
        skewDF_RF.loc[:, cat_cols_temp] = xDF_cat

    else:
        print('No missing values in categorical variables; skipped.')

# 4. Final check
t_elapsed = time.time() - t_start

print('===============================================================')
print('Imputation finished:', time.ctime())
print('Elapsed time:',
      round(t_elapsed // 60), 'min',
      round(t_elapsed % 60, 1), 'sec')

print('Original NaN:', skewDF.isna().sum().sum())
print('Imputed NaN:', skewDF_RF.isna().sum().sum())

display(
    skewDF_RF.isna().sum()
    .sort_values(ascending=False)
    .to_frame('NaN_count')
    .query('NaN_count > 0'))

In [ ]:
tempDF = skewDF_RF

#Check
count = tempDF.isnull().to_numpy().sum(axis=None)
nrows, ncols = tempDF.shape
print('nrows:', nrows)
print('ncols:', ncols)
print('nrows*ncols:', nrows*ncols)
print('nNaNs:', count, '(', count/(nrows*ncols)*100, '[%])')
print('Mean nNaNs per row:', count/nrows)
print('Mean nNaNs per column:', count/ncols)

# 7. dummy category

one-hotエンコーディングでダミー変数を作る  
多重共線性によるダミー変数トラップを回避するためにdrop_first=Trueを指定

In [ ]:
tempDF = skewDF_RF.copy()

# categorical
cat_cols = ['尿蛋白', '尿糖','食後区分']

# 1 = -/空腹 :reference
#one-hot encoding
tempDF = pd.get_dummies(
    tempDF,
    columns=cat_cols,
    prefix=cat_cols,
    drop_first=True,
    dtype=int)

display(tempDF)
skewDF_RF = tempDF

尿蛋白の4.0ないらしい

# 8. age related features

In [ ]:
tempDF1 = skewDF_RF
tempDF2 = cohortDF


# 対象データ
x = tempDF2['Age']

corr = tempDF1.corrwith(x)
japanize_matplotlib.japanize()
columns = corr.abs().sort_values(ascending=False).index
n_cols = 5
n_rows = math.ceil(len(columns) / n_cols)

fig, axes = plt.subplots(
    n_rows, n_cols,
    figsize=(20, 4 * n_rows))

axes = axes.flatten()

for i, col in enumerate(columns):
    ax = axes[i]

    # 欠損値を除外して描画
    plotDF = pd.DataFrame({
        'Age': x,
        'Feature': tempDF1[col]}).dropna()

    ax.scatter(
        plotDF['Age'],
        plotDF['Feature'],
        alpha=0.5,
        s=10)

    ax.set_title(col, fontsize=10)
    ax.set_xlabel('Age')
    ax.set_ylabel(col)
    ax.grid(alpha=0.2)

# 余ったサブプロットを削除
for j in range(len(columns), len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

GFR区分は計算に年齢を含んでいるためこれは除外する必要がある。  
γ-GTPは実測値のためよし  

In [ ]:
# updata dictionally
spe_dict = {
    'anthopometry': [
        #anthopometry
        '身長 [cm]',
        '体重 [kg]',
        '腹囲 [cm]',
        'BMI [kg/m^2]'],

    'blood_pressure':[
        #cardiovascular
        '収縮期血圧 1回目 [mmHg]',
        #'収縮期血圧 2回目 [mmHg]',
        '拡張期血圧 1回目 [mmHg]',
        #'拡張期血圧 2回目 [mmHg]'
    ],

    
    'hematopology':[
        '赤血球数 [10^4/μL]',
        '血色素量 [g/dL]',
        'ヘマトクリット値 [%]',
        '白血球数 [/μL]',
        '血小板数 [10^4/μL]',
        'リンパ球 [%]',
        '単球 [%]',
        '好酸球 [%]',
        '好塩基球 [%]',
        '好中球 [%]'],
    
    'clinical_chemistry': [
        #metabolism
        'HbA1c（NGSP） [%]',
        'グルコース [mg/dL]',
        #'総コレステロール',
        '中性脂肪 [mg/dL]',
        'HDLコレステロール [mg/dL]',
        #'LDLコレステロール（直接法） [mg/dL]',
        #liver
        'AST（GOT） [U/L]',
        'ALT（GPT） [U/L]',
        'γ-GTP [U/L]'],

    'kidney':[
        #renal(maybe from blood)
        '尿素窒素 [mg/dL]',
        '尿酸 [mg/dL]',
        'クレアチニン（血中） [mg/dL]',
        #'GFR区分（クレアチニン）', age-linear
        #renal
        '尿蛋白_2.0',
        '尿蛋白_3.0',
        #'尿蛋白_4.0',
        '尿蛋白_5.0',
        '尿糖_2.0',
        '尿糖_3.0',
        '尿糖_4.0',
        '尿糖_5.0',
        #'尿潜血'
    ],

    'inspection_conditions':[
        '食後区分_2.0']}


## -2 feature pairplot

In [ ]:
sns.set(style='ticks', context='notebook')
japanize_matplotlib.japanize()

tempDF = skewDF_RF
tempL = (spe_dict['hematopology']+spe_dict['clinical_chemistry']+spe_dict['kidney'])

# 欠損値を除外
plotDF = tempDF[tempL].dropna(axis=0)

# PairGrid
g = sns.PairGrid(plotDF, height=2)

# 左下：散布図
g.map_lower(
    sns.scatterplot,
    alpha=0.5,
    s=15)

# 対角線：分布
g.map_diag(
    sns.histplot,
    kde=True)

# 右上：相関係数とp値
def corrfunc(x, y, **kwargs):
    # 有限値のみ使用
    valid = np.isfinite(x) & np.isfinite(y)
    x = np.asarray(x)[valid]
    y = np.asarray(y)[valid]

    ax = plt.gca()

    if len(x) < 3 or np.std(x) == 0 or np.std(y) == 0:
        ax.text(
            0.5, 0.5, 'N/A',
            ha='center', va='center',
            transform=ax.transAxes)
        return

    r, p = pearsonr(x, y)

    p_text = f'p = {p:.2e}' if p < 0.001 else f'p = {p:.3f}'

    ax.annotate(
        f'r = {r:.3f}\n{p_text}\nn = {len(x)}',
        xy=(0.5, 0.5),
        xycoords='axes fraction',
        ha='center',
        va='center',
        fontsize=10)

    ax.set_axis_off()

g.map_upper(corrfunc)

g.fig.tight_layout()
plt.show()

In [ ]:
sns.set(style='ticks', context='notebook')
japanize_matplotlib.japanize()

tempDF = skewDF_RF
tempL = (spe_dict['anthopometry'])

# 欠損値を除外
plotDF = tempDF[tempL].dropna(axis=0)

# PairGrid
g = sns.PairGrid(plotDF, height=2)

# 左下：散布図
g.map_lower(
    sns.scatterplot,
    alpha=0.5,
    s=15)

# 対角線：分布
g.map_diag(
    sns.histplot,
    kde=True)

# 右上：相関係数とp値
def corrfunc(x, y, **kwargs):
    # 有限値のみ使用
    valid = np.isfinite(x) & np.isfinite(y)
    x = np.asarray(x)[valid]
    y = np.asarray(y)[valid]

    ax = plt.gca()

    if len(x) < 3 or np.std(x) == 0 or np.std(y) == 0:
        ax.text(
            0.5, 0.5, 'N/A',
            ha='center', va='center',
            transform=ax.transAxes)
        return

    r, p = pearsonr(x, y)

    p_text = f'p = {p:.2e}' if p < 0.001 else f'p = {p:.3f}'

    ax.annotate(
        f'r = {r:.3f}\n{p_text}\nn = {len(x)}',
        xy=(0.5, 0.5),
        xycoords='axes fraction',
        ha='center',
        va='center',
        fontsize=10)

    ax.set_axis_off()

g.map_upper(corrfunc)

g.fig.tight_layout()
plt.show()

In [ ]:
# updata dictionally
spe_dict = {
    'anthopometry': [
        #anthopometry
        '身長 [cm]',
        '体重 [kg]',
        #'腹囲 [cm]',
        'BMI [kg/m^2]'],

    'blood_pressure':[
        #cardiovascular
        '収縮期血圧 1回目 [mmHg]',
        #'収縮期血圧 2回目 [mmHg]',
        '拡張期血圧 1回目 [mmHg]',
        #'拡張期血圧 2回目 [mmHg]'
    ],

    
    'hematopology':[
        #'赤血球数 [10^4/μL]',
        '血色素量 [g/dL]',
        'ヘマトクリット値 [%]',
        '白血球数 [/μL]',
        '血小板数 [10^4/μL]',
        'リンパ球 [%]',
        '単球 [%]',
        '好酸球 [%]',
        '好塩基球 [%]',
        #'好中球 [%]'
    ],
    
    'clinical_chemistry': [
        #metabolism
        'HbA1c（NGSP） [%]',
        'グルコース [mg/dL]',
        #'総コレステロール',
        '中性脂肪 [mg/dL]',
        'HDLコレステロール [mg/dL]',
        #'LDLコレステロール（直接法） [mg/dL]',
        #liver
        'AST（GOT） [U/L]',
        'ALT（GPT） [U/L]',
        'γ-GTP [U/L]'],

    'kidney':[
        #renal(maybe from blood)
        '尿素窒素 [mg/dL]',
        '尿酸 [mg/dL]',
        'クレアチニン（血中） [mg/dL]',
        #'GFR区分（クレアチニン）', age-linear
        #renal
        '尿蛋白_2.0',
        '尿蛋白_3.0',
        #'尿蛋白_4.0',
        '尿蛋白_5.0',
        '尿糖_2.0',
        '尿糖_3.0',
        '尿糖_4.0',
        '尿糖_5.0',
        #'尿潜血'
    ],

    'inspection_conditions':[
        '食後区分_2.0']}


# 8. save

In [ ]:
#Save cohort metadata

fileDir = './ExportData/Preprocess/specific/'
ipynbName = '260930_TMM-BAge-CommCohort67K-ver1_'
fileName = 'demographicsDF.tsv'
tempDF = cohortDF
tempDF.to_csv(fileDir+ipynbName+fileName, sep='\t', index=True)

display(tempDF)
display(tempDF.describe(include='all'))

## -1 specific model

In [ ]:
#Save data with NAs
fileDir = './ExportData/Preprocess/specific/'
ipynbName = '260930_TMM-BAge_SpeAge_'
fileName = 'specificDF-w-NAs.tsv'

tempDF = skewDF
tempDF.to_csv(fileDir+ipynbName+fileName, sep='\t', index=True)

display(tempDF)
display(tempDF.describe(include='all'))

In [ ]:
#Save data with NAs
fileDir = './ExportData/Preprocess/specific/'
ipynbName = '260930_TMM-BAge_SpeAge_'
fileName = 'specificDF-MF.tsv'

tempDF = skewDF_RF
tempDF.to_csv(fileDir+ipynbName+fileName, sep='\t', index=True)

display(tempDF)
display(tempDF.describe(include='all'))

## -2 anthopometry

In [ ]:
#Save data with NAs
fileDir = './ExportData/Preprocess/specific/'
ipynbName = '260930_TMM-BAge_SpeAge_'
fileName = 'antDF-MF.tsv'

tempL = (spe_dict['anthopometry'])
tempD = skewDF_RF.columns.intersection(tempL)

tempDF = skewDF_RF[tempD].copy()
tempDF.to_csv(fileDir+ipynbName+fileName, sep='\t', index=True)

display(tempDF)
display(tempDF.describe(include='all'))

## -3 hemotopology

In [ ]:
#Save data with NAs
fileDir = './ExportData/Preprocess/specific/'
ipynbName = '260930_TMM-BAge_SpeAge_'
fileName = 'hemDF-MF.tsv'

tempL = (spe_dict['hematopology']+spe_dict['inspection_conditions'])
tempD = skewDF_RF.columns.intersection(tempL)

tempDF = skewDF_RF[tempD].copy()
tempDF.to_csv(fileDir+ipynbName+fileName, sep='\t', index=True)

display(tempDF)
display(tempDF.describe(include='all'))

## -4 clinical_chemistry

In [ ]:
#Save data with NAs
fileDir = './ExportData/Preprocess/specific/'
ipynbName = '260930_TMM-BAge_SpeAge_'
fileName = 'chemDF-MF.tsv'

tempL = (spe_dict['clinical_chemistry']+spe_dict['kidney']+spe_dict['inspection_conditions'])
tempD = skewDF_RF.columns.intersection(tempL)

tempDF = skewDF_RF[tempD].copy()
tempDF.to_csv(fileDir+ipynbName+fileName, sep='\t', index=True)

display(tempDF)
display(tempDF.describe(include='all'))

## -5 labotest

In [ ]:
#Save data with NAs
fileDir = './ExportData/Preprocess/specific/'
ipynbName = '260930_TMM-BAge_SpeAge_'
fileName = 'labotestDF-MF.tsv'

tempL = (spe_dict['hematopology']+spe_dict['clinical_chemistry']+spe_dict['kidney']+spe_dict['inspection_conditions'])
tempD = skewDF_RF.columns.intersection(tempL)

tempDF = skewDF_RF[tempD].copy()
tempDF.to_csv(fileDir+ipynbName+fileName, sep='\t', index=True)

display(tempDF)
display(tempDF.describe(include='all'))

## -6 kidney

In [ ]:
#Save data with NAs
fileDir = './ExportData/Preprocess/specific/'
ipynbName = '260930_TMM-BAge_SpeAge_'
fileName = 'KidDF-MF.tsv'

tempL = (spe_dict['kidney']+spe_dict['inspection_conditions'])
tempD = skewDF_RF.columns.intersection(tempL)

tempDF = skewDF_RF[tempD].copy()
tempDF.to_csv(fileDir+ipynbName+fileName, sep='\t', index=True)

display(tempDF)
display(tempDF.describe(include='all'))

# end of notebook